### Method 1: Direct File Upload

This method is suitable for smaller files or when you only need to upload a few files. The files will be deleted when the Colab runtime restarts or disconnects.

In [8]:
from google.colab import files

# This will open a file selection dialog.
# Once you select a file, it will be uploaded to the Colab environment's /content/ directory.
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')


Saving customer_shopping_behavior.csv to customer_shopping_behavior (1).csv
User uploaded file "customer_shopping_behavior (1).csv" with length 416506 bytes


After uploading, you can read the file using pandas (or other libraries), assuming it's a CSV:


In [9]:
import pandas as pd
import io
df = pd.read_csv('customer_shopping_behavior.csv')
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [11]:
df.describe()


,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [12]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [13]:
# filling the review rating blank values with the median of the product category
df['Review Rating'] =df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))


In [14]:
df.isnull().sum()

,0
Customer ID,0
Age,0
Gender,0
Item Purchased,0
Category,0
Purchase Amount (USD),0
Location,0
Size,0
Color,0
Season,0


In [15]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [16]:
#create a column age_group
labels = ['Youth', 'Young Adult', 'Adult', 'Senior']
df['age_group'] = pd.qcut(df['age'],q=4,labels=labels)

In [17]:
df[['age','age_group']].head(10)

,age,age_group
0,55,Adult
1,19,Youth
2,50,Adult
3,21,Youth
4,45,Adult
5,46,Adult
6,63,Senior
7,27,Youth
8,26,Youth
9,57,Adult


In [18]:
#create a new column for purchase_frequency_days
frequency_mapping = {'Fortnightly' : 14,
                     'Weekly' : 7,
                     'Monthly' : 30,
                     'Quarterly':90,
                     'Bi-Weekly':14,
                     'Annually':365,
                     'Every 3 Months':90}
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

In [19]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [20]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [21]:
df = df.drop('promo_code_used', axis=1)
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount_(usd)', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

In [22]:
pip install pymysql sqlalchemy


In [25]:
from sqlalchemy import create_engine
import urllib.parse

# MySQL Connection details
username = "root"
password = "Sandysony@16"
host = "localhost" # This will refer to the Colab VM, not your local machine
port = "3306"
database = "Customer_behavior"

# URL-encode the password, especially if it contains special characters like '@'
encoded_password = urllib.parse.quote_plus(password)

# Create the SQLAlchemy engine with proper host:port and encoded password
# Note: This will only work if a MySQL server is running and accessible at 'host:port' from the Colab environment.
engine = create_engine(f"mysql+pymysql://{username}:{encoded_password}@{host}:{port}/{database}")

# Write dataframe to MySQL
table_name = "customer"

try:
    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"DataFrame successfully written to MySQL table '{table_name}'")
except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure your MySQL server is running and accessible from the Colab environment, and that the connection details (host, port, username, password, database) are correct.")

An error occurred: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([Errno 111] Connection refused)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Please ensure your MySQL server is running and accessible from the Colab environment, and that the connection details (host, port, username, password, database) are correct.
